# 🛡️ Errors — advanced concepts

**Time:** ~30 minutes &nbsp;•&nbsp; **Goal:** handle messy data like a professional — your **own** error types, the pythonic "try it and see" style, and a batch job that **never dies** halfway.

This is the notebook that sets up the tech session. Soon you'll point Python at a **folder of 100 sequence files** — and some will be broken. A single crash must not throw away the other 99 results. Here you build exactly that resilience.

Skills:

1. `else` / `finally` — the full `try` shape
2. Custom exception types — errors that mean something (a peek at classes!)
3. EAFP — the pythonic "ask forgiveness" style
4. **Finale:** batch-translate a messy dataset without dying 🏆

## Our data

Run this first. Same tools as the basics notebook, plus a small **batch** of sequences — some good, some broken — like a folder of real files.

In [ ]:
# The genetic code: every three-letter codon -> its amino acid (one letter).
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
stop_codons = {"TAA", "TAG", "TGA"}

# The same short gene, sampled from two related species.
ancestor  = "ATGAAAGGTTTTCTGGAATATGCAGGCTAA"   # species A - the demos use this
species_b = "ATGAAAGGCTTCCTGGAGTATGCAAGCTAA"   # species B - your DIY challenges use this

print("species A:", ancestor)
print("species B:", species_b)


# our translate() from the functions notebooks
def translate(seq, frame=0):
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

# ...and some MESSY real-world sequences to break it on
messy = "ATGAANGGTTTT"        # an 'N' - the sequencer wasn't sure of this base
lower = "atgaaaggttttctg"     # lower-case letters


# a batch of sequences - as if read from a folder of files. Some are broken.
batch = {
    "species_A":  ancestor,
    "species_B":  species_b,
    "sample_03":  "ATGAANGGTTTT",       # an N base
    "sample_04":  "ATGAAAGG",           # length not a multiple of 3
    "sample_05":  "ATGTGGCATCCGTAA",    # fine
    "sample_06":  "ATG_CORRUPTED",      # junk from a bad download
}

## Skill 1 — `else` and `finally`: the full `try`

`try` / `except` has two more optional blocks:

```python
try:
    protein = translate(seq)
except KeyError:
    print("bad codon")
else:
    print("worked:", protein)   # runs ONLY if there was no error
finally:
    print("done with this one") # runs ALWAYS, error or not
```

- `else` — the "it worked" path, kept separate from the risky line
- `finally` — clean-up that must happen **no matter what** (closing a file, logging)

In [ ]:
# DEMO - the full try / except / else / finally
def report(seq):
    try:
        protein = translate(seq)
    except KeyError:
        print("  could not translate - unknown codon")
    else:
        print("  protein:", protein)
    finally:
        print("  (finished this sequence)")

print("clean gene:");  report(ancestor)
print("messy gene:");  report(messy)

### 🛡️ DIY challenge

1. Call `report(species_b)` and `report(lower)`.
2. For each, note which of the `else` / `finally` lines printed.
3. In a comment: which block ran for **both** the working *and* the failing sequence? Why is that useful for clean-up?

In [ ]:
# TODO - your turn

## Skill 2 — Your own exception types (a peek at classes)

Built-in errors are generic. You can define one that says exactly what went wrong — and it's your **first class**:

```python
class NotDNAError(ValueError):
    """Raised when a sequence contains non-DNA characters."""
```

That's it — one line inheriting from `ValueError`. Now you can `raise NotDNAError(...)` and callers can `except NotDNAError` specifically, while it still behaves like the `ValueError` it is. (Full classes are the *next* lecture — this is the teaser.)

In [ ]:
# DEMO - a custom error that means one specific thing
class NotDNAError(ValueError):
    """Raised when a sequence contains characters other than A/T/G/C."""

def clean_dna(seq: str) -> str:
    seq = seq.upper().strip()
    for base in seq:
        if base not in "ATGC":
            raise NotDNAError(f"{seq!r} contains {base!r}, which isn't DNA")
    return seq

try:
    clean_dna("ATG_CORRUPTED")
except NotDNAError as error:
    print("caught our own error:", error)

### 🛡️ DIY challenge

1. Define a second custom error, `class BadLengthError(ValueError):` with a docstring.
2. Write `check_length(seq)` that raises `BadLengthError` when `len(seq) % 3 != 0`, else returns `seq`.
3. Wrap a call on `"ATGAAAGG"` in `try` / `except BadLengthError as e:` and print the message.

In [ ]:
# TODO - your turn

## Skill 3 — EAFP: the pythonic "ask forgiveness" style

Two ways to handle a maybe-bad codon:

```python
# LBYL - "Look Before You Leap": check first
if codon in codon_table:
    aa = codon_table[codon]

# EAFP - "Easier to Ask Forgiveness than Permission": just try it
try:
    aa = codon_table[codon]
except KeyError:
    aa = "X"
```

Python **prefers EAFP** — try the thing, handle the failure if it comes. It's usually cleaner, and it avoids a race where the check passes but the action still fails. Here `X` is the standard "unknown amino acid" symbol.

In [ ]:
# DEMO - EAFP inside translate: unknown codons become 'X' instead of crashing
def translate_tolerant(seq):
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        try:
            protein = protein + codon_table[codon]   # just try the lookup
        except KeyError:
            protein = protein + "X"                   # unknown base -> X
    return protein

print("messy sequence:", translate_tolerant(messy))   # the N becomes X

### 🛡️ DIY challenge

1. Run `translate_tolerant` on `ancestor` (clean) and on `"ATGNNNTTT"`.
2. Where do the `X`s appear, and where don't they?
3. Rewrite the lookup in the **LBYL** style (`if codon in codon_table:` … `else:`) so it gives the same result. Which version do you find easier to read?

In [ ]:
# TODO - your turn

## Skill 4 — Finale: a batch job that never dies 🏆

The real-world skill: process a whole **batch**, and when one item is broken, **record it and move on** — never let one bad file kill the run. This is precisely what you'll do to a folder of 100 files in the tech session.

Collect the wins in one dict and the failures in another, so at the end you have *both* your results **and** a report of what went wrong.

In [ ]:
# DEMO - translate a whole batch; survive the broken ones
class BadLengthError(ValueError):
    """Raised when a sequence length isn't a whole number of codons."""

def translate_batch(sequences: dict) -> dict:
    """Translate every sequence; separate the successes from the failures."""
    results = {}
    failures = {}
    for name, seq in sequences.items():
        try:
            clean = clean_dna(seq)          # raises NotDNAError on junk
            if len(clean) % 3 != 0:
                raise BadLengthError(f"length {len(clean)} isn't a multiple of 3")
            results[name] = translate(clean)
        except ValueError as error:         # catches NotDNAError AND BadLengthError
            failures[name] = str(error)
    return {"results": results, "failures": failures}

report = translate_batch(batch)
print("translated OK:")
for name, protein in report["results"].items():
    print(f"  {name}: {protein}")
print("could not translate:")
for name, why in report["failures"].items():
    print(f"  {name}: {why}")

> This DEMO reuses `clean_dna` from Skill 2 — run that cell first. It defines its own `BadLengthError` at the top, so the two custom errors are caught together by one `except ValueError`.

### 🛡️ DIY challenge

1. Add two more sequences to `batch` — one valid, one broken in a new way — and re-run `translate_batch`.
2. Print how many sequences succeeded and how many failed (`len(report["results"])`, `len(report["failures"])`).
3. Why is returning **both** dicts better than just printing errors as they happen? (Think about what you'd want after running this on 100 files overnight.)

In [ ]:
# TODO - your turn

## 🔭 Explore further

*(Helfrid will drop free web resources here.)*

- `logging` instead of `print` — real programs *log* their failures with timestamps and severity. **[link]**
- The `with` statement (context managers) — the clean way to open files so they *always* close, even on error. You'll meet it in the tech session. **[link]**
- `warnings` — for "this is odd but not fatal" messages that don't stop the program. **[link]**

## ✅ You've got it if you can…

- [ ] Use `else` and `finally` alongside `try` / `except`
- [ ] Define a custom exception with `class MyError(ValueError):` and `raise` it
- [ ] Explain EAFP vs LBYL, and why Python leans EAFP
- [ ] Write a batch loop that collects successes and failures separately
- [ ] Explain why one broken file shouldn't kill a 100-file run

**Next up — the tech session:** point Python at a real **folder** of sequence files with `pathlib`, and run your translator across a whole evolutionary dataset.